In [ ]:
import os
from mistralai import Mistral
import numpy as np
import json
import faiss
import math
# --- CONFIG ---
client = Mistral(api_key="")
embed_model = "mistral-embed"
llm_model = "mistral-large-latest"

In [ ]:
def get_embedding(text):
    emb = client.embeddings.create(
        model=embed_model,
        inputs=[text]
    )
    return emb.data[0].embedding   # ← LISTE, pas ndarray

In [ ]:
with open("composants_techniques.json", "r") as f:
    sympt_dict = json.load(f)

In [ ]:
embedding_dict = {}

for code, data in sympt_dict.items():
    mots = data["mots_cles"]
    joined = ", ".join(mots)

    embedding = get_embedding(joined)  # LIST

    embedding_dict[code] = {
        "titre": data["titre"],
        "mots_cles": mots,
        "embedding": embedding
    }

with open("composants_techniques.json", "w") as f:
    json.dump(embedding_dict, f, indent=2)

print("Fichier symptomes_embeddings.json généré !")


In [ ]:
# Charger le JSON
with open("mots_cles_protocoles.json", "r") as f:
    sympt_dict = json.load(f)

# Reconstruire la liste des codes
codes = list(sympt_dict.keys())

# Construire la matrice d'embeddings
embeddings = np.vstack([
    np.array(sympt_dict[c]["embedding"], dtype="float32")
    for c in codes
])

contexts = [
    f"{sympt_dict[c]['titre']} — {', '.join(sympt_dict[c]['mots_cles'])}"
    for c in codes
]


In [ ]:
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

In [ ]:
def normalize_matrix_rows(M):
    """Normalise en L2 chaque ligne d'une matrice (N x D) -> (N x D)."""
    norms = np.linalg.norm(M, axis=1, keepdims=True) + 1e-12 #On ajoute 1e-12 pour éviter de diviser par 0
    return M / norms

def normalize_vec(v):
    n = np.linalg.norm(v) + 1e-12
    return v / n

In [ ]:
import re

# ---------------------------
# 1) Cosinus entre vecteurs
# ---------------------------
def cosine_similarity(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return np.dot(a, b)

# ------------------------------------------------------
# 2) RBF + softmax sur cosines pour scores globaux
# ------------------------------------------------------
def rbf_probs_from_cosines(query_vec, category_vectors, sigma=0.2, top_k=None):
    """
    Convertit cosines -> probabilités via RBF sur la distance d = 1 - cos.
    sigma: écart-type 
    top_k: si donné, ne calcule que sur top_k candidats
    Retour: (indices_topk, probs_topk, full_cosines_topk)   
    """
    q = normalize_vec(np.array(query_vec, dtype=np.float32))
    V = np.array(category_vectors, dtype=np.float32) 
    Vn = normalize_matrix_rows(V)     # normaliser lignes de V 


    
    cosines = Vn.dot(q)   # produits scalaires rapides

    if top_k is not None and top_k < len(cosines):
        top_idxs = np.argpartition(-cosines, top_k-1)[:top_k] #On récupère les k plus grands produits scalaires
        
        top_idxs = top_idxs[np.argsort(-cosines[top_idxs])] #On les range dans l'ordre croissant
        top_cos = cosines[top_idxs]
    else:
        top_idxs = np.arange(len(cosines))
        top_cos = cosines

    d = 1.0 - top_cos #On utilise plutôt 1-cos de sorte à ce que la proba vale 1 pour 2 vecteurs identiques

    # RBF kernel: exp( - d^2 / (2 * sigma^2) )
    denom = 2.0 * (sigma**2) + 1e-12
    weights = np.exp(-(d**2) / denom)


    probs = weights / (np.sum(weights) + 1e-12) #On normalise et ajoute 1e-12 pour ne pas diviser par 0

    return top_idxs, probs.astype(float), top_cos

In [ ]:

# 3) RETRIEVAL FUNCTION

def retrieve(query, k=3):
    # embed la requête
    q_emb = client.embeddings.create(
        model=embed_model,
        inputs=[query]
    ).data[0].embedding

    q_emb = np.array([q_emb], dtype="float32")

    # recherche FAISS
    D, I = index.search(q_emb, k)

    return [contexts[i] for i in I[0]]

In [ ]:
def rag_answer_expert(query, client, llm_model, k=5):
    retrieved = retrieve(query, k)
    context = "\n\n".join(retrieved)

    prompt = f"""
[INST] Tu es la mémoire technique et scientifique du Laboratoire Kastler Brossel (LKB). Ton rôle est d'aider les chercheurs en répondant à leurs questions à partir de deux sources : des articles scientifiques (théorie, protocoles) et des fiches techniques (composants, électronique, optique).

Voici les extraits pertinents retrouvés dans la base de connaissances :
{context}

Question du chercheur : {query}

CONSIGNES DE RÉPONSE ET RIGUEUR :
1. **Jongler entre Théorie et Pratique :** - Si la question porte sur un choix de composant ou une panne, appuie-toi en priorité sur les fiches techniques (références, tolérances, câblage).
   - Si la question porte sur le protocole, les calculs ou la physique de la manip, appuie-toi sur les articles scientifiques.
2. **Vocabulaire du Labo :** Reste très rigoureux et utilise le jargon exact du LKB et de la physique quantique (ex: laser de repompage, refroidissement Doppler, cavité Fabry-Perot, etc.).
3. **Citer les Sources :** Pour chaque affirmation importante, indique entre parenthèses s'il s'agit d'une spécification constructeur (ex: [Fiche Technique - Newport Oris]) ou d'un résultat publié (ex: [Article - Phys. Rev. Lett. 2023]).
4. **Honnêteté Scientifique :** Si les fiches techniques ou les articles fournis ne permettent pas de répondre précisément à la configuration de la manip (ex: manque la valeur d'une résistance ou une longueur d'onde exacte), dis-le. Ne devine jamais une spécification technique.

Format attendu : Une réponse rédigée de niveau doctorat, structurée avec des sous-titres si nécessaire, suivie de la liste des sources.
[/INST]
"""

    response = client.chat.complete(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content